# Bronze Layer: Ingestion Pipeline

This notebook implements an automated ingestion pipeline using **Databricks Auto Loader** (`cloudFiles`) combined with a `.trigger(availableNow=True)` policy to process incremental data efficiently on serverless compute.

### Key Architectural Features:
* **Automated Table Creation & Schema Evolution:** Automatically provisions target tables if they do not exist and safely handles incoming schema changes using `addNewColumns` mode.
* **External Delta Tables:** All raw Bronze tables are stored externally as Delta tables within designated S3 storage paths.
* **Metadata Tracking:** Captures operational metadata (`file_name`, `file_path`, `file_modification_time`, and `processing_timestamp`) for end-to-end lineage tracking.
* **Stateful Resilience:** Relies on dedicated S3 locations for Auto Loader checkpoints and schema inference to ensure fault tolerance and exactly-once processing guarantees.
* **Parametrized Catalog Integration:** Designed to accept catalog names dynamically as external job variables for seamless multi-environment deployments (Dev, Test, Prod).

In [0]:
%sql

DROP TABLE IF EXISTS bakery_pos_develop.bronze.account_move;
DROP TABLE IF EXISTS bakery_pos_develop.bronze.account_move_line;
DROP TABLE IF EXISTS bakery_pos_develop.bronze.pos_order;
DROP TABLE IF EXISTS bakery_pos_develop.bronze.pos_order_line;
DROP TABLE IF EXISTS bakery_pos_develop.bronze.account_journal;
DROP TABLE IF EXISTS bakery_pos_develop.bronze.res_users;
DROP TABLE IF EXISTS bakery_pos_develop.bronze.res_partner;
DROP TABLE IF EXISTS bakery_pos_develop.bronze.account_account;
DROP TABLE IF EXISTS bakery_pos_develop.bronze.pos_session;
DROP TABLE IF EXISTS bakery_pos_develop.bronze.product_product;

In [0]:
%fs rm -r 's3://bakery-pos-bk-978430483777-us-east-2-an/checkpointLocation'

In [0]:
%fs rm -r 's3://bakery-pos-bk-978430483777-us-east-2-an/schemaLocation'

In [0]:
%fs rm -r 's3://bakery-pos-bk-978430483777-us-east-2-an/checkpointLocation'

In [0]:

from pyspark.sql import functions as F

def load_bronze_layer(catalog, table, year, day, month):
    schemaLocation = f"s3://bakery-pos-bk-978430483777-us-east-2-an/schemaLocation/{table}"
    #input_path = f"s3://bakery-pos-bk-978430483777-us-east-2-an/bakery-pos/historical/20261022194601/{table}_records"
    input_path = f"s3://bakery-pos-bk-978430483777-us-east-2-an/bakery-pos/{year}/{month}/{day}/{table}"
    checkpointLocation = f"s3://bakery-pos-bk-978430483777-us-east-2-an/checkpointLocation/{table}"
    tableLocation = f"s3://bakery-pos-bk-978430483777-us-east-2-an/devel_bronze_layer/{table}"
    
    files = dbutils.fs.ls(input_path)
    if not files:
        print(f"No files found in {input_path}")
        return

    raw_df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", schemaLocation)
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .option("multiLine", "true")
        .load(input_path)
        .select(
            "*",
            F.col("_metadata.file_name").alias("file_name"),
            F.col("_metadata.file_path").alias("file_path"),
            F.col("_metadata.file_modification_time").alias("file_modification_time"),
            F.current_timestamp().alias("processing_timestamp")
        )
    )

    raw_df.printSchema()

    query = (
        raw_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpointLocation)
        .option("mergeSchema", "true")
        .option("path", tableLocation) 
        .trigger(availableNow=True)
        .table(f"{catalog}.bronze.{table}")
    )

    query.awaitTermination()

---

### Batch Execution: Processing Tables in the Schema

The loop below iterates through the core operational tables of the Odoo POS system, invoking the ingestion function dynamically for each dataset.

In [0]:
import datetime
import pytz

bogota_tz = pytz.timezone("America/Bogota")

year = datetime.datetime.now(tz=bogota_tz).strftime("%Y")
day = datetime.datetime.now(tz=bogota_tz).strftime("%d")
month = datetime.datetime.now(tz=bogota_tz).strftime("%m")

tables = [
    "account_account",
    "account_journal",
    "account_move",
    "account_move_line",
    "pos_order",
    "pos_order_line",
    "pos_session",
    "product_product",
    "res_partner",
    "res_users"
]

for table in tables:
    try:
        print(f"Processing {table}...")
        load_bronze_layer("bakery_pos_develop", table, year, day, month)
        print(f"Successfully loaded {table}")
    except Exception as e:
        print(f"Error loading {table}: {str(e)}")
        continue

In [0]:
%sql

SELECT * FROM bakery_pos_develop.bronze.account_move

In [0]:
%sql

SELECT * FROM bakery_pos_develop.bronze.res_users
 limit 10;
 
-- ORDER BY date
 -- DESC;